# Presentation figure: patient characteristics

One slide figure for "do sex or age at surgery go with improvement?", both outcomes, read top to bottom: the observed data first, then the model that tests them. Colour means the outcome throughout: **teal Engel, orange ILAE**.

- **A. Age at surgery (observed):** all patients, improved and not improved, Engel and ILAE side by side, with n and median (the "all" dots in grey).
- **B. Improvement by sex (observed):** male and female patients, Engel and ILAE, with Wilson 95% CIs and the counts.
- **C. Odds of improvement:** Firth odds ratios for male sex and age at surgery (per year), Engel and ILAE, with Holm p: the one test per question on this slide.

The sample sizes are in A and B, so C doesn't repeat them; the separate tests of the same questions (Mann-Whitney for age, Fisher's exact for sex) agree with the models and are in `0_cohort/patient_characteristics.ipynb`, which also runs the models; this notebook reads its results (`firths_results_covariates.csv`), so run `pixi run run-analysis` first. Saved as `results/<version>/presentation/patient_characteristics.png`. The figure is drawn at slide size (10 x 4.9 in, `cc_stats.plots.SLIDE_SIZE`): place it full width on a 16:9 Google Slides page, below the slide title, and its text keeps its point size.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cc_stats.outcomes import OUTCOMES
from cc_stats.paths import ANALYSIS_CSV, DATASET, results_dir
from cc_stats.plots import (INK, PAIR_A, PAIR_B, forest, paired_rate_plot, right_of, save_presentation_figure,
                            setup_or_axis, slide_figure)

print(f'dataset {DATASET}')
analysis = pd.read_csv(ANALYSIS_CSV, index_col='subject_id')
cov = pd.read_csv(results_dir('0_cohort') / 'firths_results_covariates.csv')
# The cohort of the Firth covariate models: the outcome, sex and age recorded
covariates = {o: analysis.dropna(subset=[O['improved'], 'sex', 'age_at_cc']) for o, O in OUTCOMES.items()}
COLOR = {'engel': PAIR_A, 'ilae': PAIR_B}


def fmt_p(v):
    return '<0.001' if v < 0.001 else f'{v:.3f}'

In [ ]:
fig = slide_figure()
grid = fig.add_gridspec(2, 2, height_ratios=[1.35, 1], width_ratios=[1.3, 1])
age_ax, sex_ax = fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1])
or_ax = fig.add_subplot(grid[1, :])

# A: age at surgery, all / improved / not improved, Engel (left three) and ILAE (right three)
rng = np.random.default_rng(7)
positions, labels = [], []
for j, (outcome, O) in enumerate(OUTCOMES.items()):
    c = covariates[outcome]
    for i, (name, improved) in enumerate((('all', None), ('improved', 1), ('not\nimproved', 0))):
        x = j * 3.5 + i
        v = c['age_at_cc'] if improved is None else c.loc[c[O['improved']] == improved, 'age_at_cc']
        age_ax.boxplot(v, positions=[x], widths=0.6, showfliers=False, medianprops=dict(color=INK, lw=1.8))
        age_ax.scatter(x + rng.uniform(-0.15, 0.15, len(v)), v, s=7, alpha=0.7, zorder=3,
                       color='tab:gray' if improved is None else COLOR[outcome])
        age_ax.text(x, 1.01, f'n={len(v)}\n{v.median():.0f} y', transform=age_ax.get_xaxis_transform(),
                    ha='center', va='bottom', fontsize=9.5)
        positions.append(x)
        labels.append(name)
    age_ax.text(j * 3.5 + 1, -0.3, O['name'], transform=age_ax.get_xaxis_transform(), ha='center', va='top',
                fontsize=11, fontweight='bold', color=COLOR[outcome])
age_ax.set_xticks(positions, labels, fontsize=9.5)
age_ax.set_xlim(-0.6, 6.1)
age_ax.set_ylim(0, None)
age_ax.set_ylabel('Age at surgery (years)')
age_ax.set_title('A.  Age at surgery (n, median)', loc='left', pad=26)
age_ax.spines[['top', 'right']].set_visible(False)

# B: observed improvement in male and female patients, Engel above ILAE
SEXES = ['Female', 'Male']  # bottom to top
k, n = {}, {}
for outcome, O in OUTCOMES.items():
    c = covariates[outcome]
    groups = {s: c.loc[c.sex == s, O['improved']] for s in SEXES}
    k[outcome] = [int(groups[s].sum()) for s in SEXES]
    n[outcome] = [len(groups[s]) for s in SEXES]
paired_rate_plot(sex_ax, [s.lower() for s in SEXES],
                 top=('Engel', k['engel'], n['engel'], PAIR_A),
                 bottom=('ILAE', k['ilae'], n['ilae'], PAIR_B), header='Improved', fontsize=10)
sex_ax.set_xticks([0, 0.5, 1])
sex_ax.set_xlabel('Improved (Wilson 95% CI)')
sex_ax.set_title('B.  Improvement by sex', loc='left', pad=26)

# C: Firth ORs for both, Engel and ILAE together, with Holm p
names = {'sex': 'sex (male)', 'age_at_cc': 'age (per year)'}
order = ['age_at_cc', 'sex']  # bottom to top
for (outcome, O), offset in zip(OUTCOMES.items(), (0.18, -0.18)):
    d = cov.query('outcome == @outcome').set_index('predictor').loc[order].reset_index()
    forest(or_ax, d, ['OR', 'lo', 'hi'], offset=offset, color=COLOR[outcome], label=O['name'])
setup_or_axis(or_ax, [names[p] for p in order], '', ticks=(0.1, 0.25, 0.5, 1, 2, 4, 10))
or_ax.set_ylim(-0.6, len(order) - 0.4)
or_ax.set_title('C.  Odds of improvement (univariable Firth models)', loc='left')
or_ax.text(1, len(order) - 0.25, 'Holm p', transform=right_of(or_ax, 0.45), va='center', ha='center',
           fontsize=10, fontweight='bold')
for (outcome, O), offset in zip(OUTCOMES.items(), (0.18, -0.18)):
    d = cov.query('outcome == @outcome').set_index('predictor')
    for y, p in enumerate(order):
        or_ax.text(1, y + offset, fmt_p(d.loc[p, 'p_holm']), transform=right_of(or_ax, 0.45), va='center',
                   ha='center', fontsize=10, color=COLOR[outcome])

# One legend for the whole figure: colour means the outcome in every panel
handles, names_ = or_ax.get_legend_handles_labels()
fig.legend(handles, names_, loc='outside upper right', ncol=2, frameon=False)
save_presentation_figure('patient_characteristics', fig)
plt.show()